# Entrenamiento de modelos - Palmer Penguins

Versión "pingüinos" del notebook de entrenamiento. Usa el dataset
**Palmer Penguins** (paquete `palmerpenguins`, viene con el CSV
incluido, no necesita internet) y entrena un `DecisionTreeClassifier`
para predecir la especie.

Guarda el modelo en el mismo formato que `train_model.ipynb`
(`model_<timestamp>.joblib` + `model_<timestamp>.json` con
`feature_names` y `target_names`) en el volumen **compartido**
(`MODELS_DIR`, montado como `./models` en el host), así que la API
(`ml_api`) lo consume automáticamente sin ningún cambio de código:
solo le importa el `.joblib` más reciente y sus metadatos, no de qué
dataset viene.

**Importante:** si ya entrenaste con `train_model.ipynb` (iris), el
modelo más reciente (por fecha) es el que la API sirve. Si entrenas
aquí después, la API pasa a usar el de pingüinos automáticamente.

In [1]:
import os
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import palmerpenguins

MODELS_DIR = os.environ.get("MODELS_DIR", "/models")
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Los modelos se guardaran en: {MODELS_DIR}")

Los modelos se guardaran en: /models


## 1. Cargar y limpiar el dataset

`palmerpenguins` trae el CSV embebido en el paquete: no necesita red.

In [2]:
df = palmerpenguins.load_penguins()

# Quitamos filas con datos faltantes en las columnas que vamos a usar
df = df.dropna(subset=[
    "species", "island", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g", "sex",
]).reset_index(drop=True)

print(df.shape)
df.head()

(333, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


## 2. Construir features (numéricas + one-hot de columnas categóricas)

`island` y `sex` son categóricas: las convertimos a columnas 0/1 con
categorías **fijas** (no con `pd.get_dummies` a secas), para que el
orden y la cantidad de columnas sean siempre los mismos sin importar
qué categorías aparezcan en la muestra de entrenamiento. Ese orden
fijo (`feature_names`) es el que después tienes que respetar al armar
el vector `features` que le mandas a `/predict`.

In [3]:
ISLANDS = ["Biscoe", "Dream", "Torgersen"]
SEXES = ["female", "male"]
SPECIES = sorted(df["species"].unique())  # target_names, orden fijo y determinista

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)
    out["bill_length_mm"] = frame["bill_length_mm"].astype(float)
    out["bill_depth_mm"] = frame["bill_depth_mm"].astype(float)
    out["flipper_length_mm"] = frame["flipper_length_mm"].astype(float)
    out["body_mass_g"] = frame["body_mass_g"].astype(float)
    for island in ISLANDS:
        out[f"island_{island}"] = (frame["island"] == island).astype(float)
    for sex in SEXES:
        out[f"sex_{sex}"] = (frame["sex"] == sex).astype(float)
    return out

X = build_features(df)
y = df["species"].map({name: i for i, name in enumerate(SPECIES)})
feature_names = list(X.columns)
target_names = SPECIES

print("feature_names (en este orden exacto):")
print(feature_names)
print()
print("target_names (indice -> especie):")
print(list(enumerate(target_names)))

X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

feature_names (en este orden exacto):
['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'island_Biscoe', 'island_Dream', 'island_Torgersen', 'sex_female', 'sex_male']

target_names (indice -> especie):
[(0, 'Adelie'), (1, 'Chinstrap'), (2, 'Gentoo')]

Train: (266, 9), Test: (67, 9)


## 3. Función de entrenamiento y guardado versionado

Mismo formato que `train_model.ipynb`: `.joblib` + `.json` con timestamp único.

In [4]:
def train_and_save(max_depth=None, min_samples_leaf=1, random_state=42):
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    timestamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
    model_filename = f"model_{timestamp}.joblib"
    meta_filename = f"model_{timestamp}.json"

    model_path = os.path.join(MODELS_DIR, model_filename)
    meta_path = os.path.join(MODELS_DIR, meta_filename)

    joblib.dump(model, model_path)

    metadata = {
        "timestamp": timestamp,
        "created_at": datetime.utcnow().isoformat() + "Z",
        "algorithm": "DecisionTreeClassifier",
        "dataset": "palmerpenguins",
        "hyperparameters": {
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "random_state": random_state,
        },
        "accuracy": accuracy,
        "feature_names": feature_names,
        "target_names": target_names,
        "model_file": model_filename,
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Modelo guardado: {model_path}")
    print(f"Metadatos guardados: {meta_path}")
    print(f"Accuracy en test: {accuracy:.4f}")
    return model_path, metadata

## 4. Entrenar el primer modelo

In [5]:
train_and_save(max_depth=None)

Modelo guardado: /models/model_20260907000705.joblib
Metadatos guardados: /models/model_20260907000705.json
Accuracy en test: 0.9254


('/models/model_20260907000705.joblib',
 {'timestamp': '20260907000705',
  'created_at': '2026-09-07T00:07:05.075500Z',
  'algorithm': 'DecisionTreeClassifier',
  'dataset': 'palmerpenguins',
  'hyperparameters': {'max_depth': None,
   'min_samples_leaf': 1,
   'random_state': 42},
  'accuracy': 0.9253731343283582,
  'feature_names': ['bill_length_mm',
   'bill_depth_mm',
   'flipper_length_mm',
   'body_mass_g',
   'island_Biscoe',
   'island_Dream',
   'island_Torgersen',
   'sex_female',
   'sex_male'],
  'target_names': ['Adelie', 'Chinstrap', 'Gentoo'],
  'model_file': 'model_20260907000705.joblib'})

## 5. Entrenar nuevos modelos (versionado)

Cada vez que corras esta celda con otros hiperparámetros se genera un
**nuevo** archivo en el volumen compartido. La API detecta el `.joblib`
más reciente en la siguiente petición a `/predict` (o con `/reload`),
sin reiniciar el contenedor.

In [6]:
# Ejemplo: un árbol más simple (menos profundidad) para comparar accuracy
train_and_save(max_depth=3)

Modelo guardado: /models/model_20260907000709.joblib
Metadatos guardados: /models/model_20260907000709.json
Accuracy en test: 0.9552


('/models/model_20260907000709.joblib',
 {'timestamp': '20260907000709',
  'created_at': '2026-09-07T00:07:09.121067Z',
  'algorithm': 'DecisionTreeClassifier',
  'dataset': 'palmerpenguins',
  'hyperparameters': {'max_depth': 3,
   'min_samples_leaf': 1,
   'random_state': 42},
  'accuracy': 0.9552238805970149,
  'feature_names': ['bill_length_mm',
   'bill_depth_mm',
   'flipper_length_mm',
   'body_mass_g',
   'island_Biscoe',
   'island_Dream',
   'island_Torgersen',
   'sex_female',
   'sex_male'],
  'target_names': ['Adelie', 'Chinstrap', 'Gentoo'],
  'model_file': 'model_20260907000709.joblib'})

## 6. Ver todos los modelos guardados en el volumen compartido

In [7]:
saved = sorted(
    [f for f in os.listdir(MODELS_DIR) if f.endswith(".joblib")],
    key=lambda f: os.path.getmtime(os.path.join(MODELS_DIR, f)),
)
for f in saved:
    mtime = time.ctime(os.path.getmtime(os.path.join(MODELS_DIR, f)))
    print(f"{f}  (modificado: {mtime})")

if saved:
    print(f"\nModelo mas reciente (el que consumira la API): {saved[-1]}")

model_20260907000705.joblib  (modificado: Mon Sep  7 00:07:05 2026)
model_20260907000709.joblib  (modificado: Mon Sep  7 00:07:09 2026)

Modelo mas reciente (el que consumira la API): model_20260907000709.joblib


## 7. Ejemplo de request a `/predict`

Usa una fila real del dataset para armar un vector `features` válido,
en el mismo orden que `feature_names` (arriba).

In [8]:
sample = X.iloc[0]
sample_features = [float(v) for v in sample.values]

print("features (en el orden de feature_names):")
print(sample_features)
print()
print("Ejemplo de curl:")
print("curl -X POST http://localhost:8000/predict \\")
print("  -H 'Content-Type: application/json' \\")
print(f"  -d '{{\"features\": {sample_features}}}'")

features (en el orden de feature_names):
[39.1, 18.7, 181.0, 3750.0, 0.0, 0.0, 1.0, 0.0, 1.0]

Ejemplo de curl:
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [39.1, 18.7, 181.0, 3750.0, 0.0, 0.0, 1.0, 0.0, 1.0]}'
